# Day 4 Capstone Lab: ESG-Aware Treasury Analytics App
In this sprint you will turn the previous labs into a toy fintech application that recommends responsible short-term investments for treasurers.

> **App scope**
> - ingest Bloomberg ESG scores + yield curves
> - compute responsible weights subject to policy constraints
> - expose SHAP-style explanations for each issuer
> - ship an interactive UI (Streamlit or Gradio) that runs in Colab

## 0. Sprint checklist
1. Finalize your data assets (reuse `DATA_ROOT` from earlier labs).
2. Decide which UI framework to demo:
   - **Gradio** (fastest, works inside Colab)
   - Streamlit (requires `pyngrok` → see helper cell at the bottom)
3. Outline user stories (e.g., "As a treasurer, I want to see compliant issuers with expected yields").
4. Split the team: data engineer, model lead, UX/documentation.

In [ ]:
%%capture
!pip install pandas numpy scikit-learn plotly==5.24.0 shap gradio==4.42.0

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import shap

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd() / "data" / "bloomberg"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)


In [ ]:
esg_path = DATA_ROOT / "esg_scores_SAMPLE.csv"
yield_path = DATA_ROOT / "yield_curve_SAMPLE.csv"

try:
    esg = pd.read_csv(esg_path)
    yields = pd.read_csv(yield_path)
    print("Loaded ESG + yield files.")
except FileNotFoundError:
    print("Files not found. Using synthetic placeholders; replace during the live sprint.")
    tickers = [f"Issuer{i:02d}" for i in range(12)]
    rng = np.random.default_rng(123)
    esg = pd.DataFrame({
        "ticker": tickers,
        "env_disclosure_score": rng.integers(40, 95, len(tickers)),
        "soc_disclosure_score": rng.integers(30, 90, len(tickers)),
        "gov_disclosure_score": rng.integers(35, 92, len(tickers)),
        "country": rng.choice(["US", "UK", "FR", "BR"], len(tickers))
    })
    yields = pd.DataFrame({
        "ticker": np.repeat(tickers, 3),
        "tenor": np.tile(["3M", "6M", "1Y"], len(tickers)),
        "yield": rng.normal(2.5, 0.4, len(tickers) * 3)
    })

def summarize_inputs(esg_df: pd.DataFrame, yield_df: pd.DataFrame) -> pd.DataFrame:
    pivot = yield_df.pivot_table(index="ticker", values="yield", aggfunc="mean")
    merged = esg_df.join(pivot, on="ticker", how="left", rsuffix="_avg")
    merged.rename(columns={"yield": "avg_yield"}, inplace=True)
    return merged.dropna(subset=["avg_yield"])

inputs = summarize_inputs(esg, yields)
inputs.head()

In [ ]:
POLICY = {
    "min_esg_score": 60,
    "max_country_weight": 0.4,
    "max_single_issuer_weight": 0.15
}

eligible = inputs[
    (inputs[["env_disclosure_score", "soc_disclosure_score", "gov_disclosure_score"]].min(axis=1) >= POLICY["min_esg_score"])
]
eligible

In [ ]:
def allocate_portfolio(df: pd.DataFrame, policy: dict, total_weight: float = 1.0) -> pd.DataFrame:
    df = df.sort_values("avg_yield", ascending=False).copy()
    df["weight"] = 0.0
    country_usage = {}
    remaining = total_weight
    for idx, row in df.iterrows():
        if remaining <= 0:
            break
        country = row["country"]
        country_cap = policy["max_country_weight"]
        issuer_cap = policy["max_single_issuer_weight"]
        current_country = country_usage.get(country, 0.0)
        available = min(issuer_cap, country_cap - current_country, remaining)
        if available <= 0:
            continue
        df.at[idx, "weight"] = available
        country_usage[country] = current_country + available
        remaining -= available
    return df[df["weight"] > 0]

portfolio = allocate_portfolio(eligible, POLICY)
portfolio

In [ ]:
feature_cols = ["env_disclosure_score", "soc_disclosure_score", "gov_disclosure_score"]
target_col = "avg_yield"

model = Pipeline([
    ("scale", StandardScaler()),
    ("rf", RandomForestRegressor(n_estimators=400, random_state=42))
])
model.fit(inputs[feature_cols], inputs[target_col])
explainer = shap.TreeExplainer(model.named_steps["rf"])
shap_values = explainer.shap_values(inputs[feature_cols])
shap.summary_plot(shap_values, inputs[feature_cols], plot_type="bar", show=False)

In [ ]:
import gradio as gr

inputs_serialized = inputs.to_dict(orient="records")


def run_recommender(env_score, soc_score, gov_score, country):
    query = pd.DataFrame([
        {
            "env_disclosure_score": env_score,
            "soc_disclosure_score": soc_score,
            "gov_disclosure_score": gov_score,
            "country": country
        }
    ])
    avg_yield_pred = model.predict(query[feature_cols])[0]
    shap_vals = explainer.shap_values(query[feature_cols])[0]
    reason = {
        "feature": feature_cols,
        "impact_bps": (shap_vals * 100).round(2)
    }
    return float(avg_yield_pred), reason

country_choices = sorted(inputs["country"].unique())

ui = gr.Interface(
    fn=run_recommender,
    inputs=[
        gr.Slider(40, 95, value=75, label="Env disclosure"),
        gr.Slider(40, 95, value=70, label="Soc disclosure"),
        gr.Slider(40, 95, value=72, label="Gov disclosure"),
        gr.Dropdown(choices=country_choices, value=country_choices[0], label="Country")
    ],
    outputs=[
        gr.Number(label="Predicted yield (%)"),
        gr.Dataframe(headers=["feature", "impact_bps"])
    ],
    title="ESG-Aware Treasury Advisor",
    description="Explore how ESG scores drive recommended yields"
)

ui.launch(share=True, height=600)

### (Optional) Streamlit + ngrok
If you prefer a Streamlit UI:
1. Edit `app/streamlit_app.py` in the repo (template provided).
2. Run the helper cell below to install `pyngrok` and expose a public URL.
3. Share the URL + screenshot in your assessment pack.

In [ ]:
%%capture
!pip install streamlit pyngrok

In [ ]:
import os
from pyngrok import ngrok

os.environ["STREAMLIT_SERVER_HEADLESS"] = "true"
os.environ["STREAMLIT_SERVER_PORT"] = "8501"
tunnel = ngrok.connect(8501)
print("Public URL:", tunnel.public_url)

!streamlit run /workspace/app/streamlit_app.py

### Deliverables
- Deployed Gradio or Streamlit demo link (screenshot if link expires).
- Model card covering data sources, fairness checks, and monitoring plan.
- Pitch slide summarising business value + responsible AI safeguards.